In [1]:
import pandas as pd
import numpy as np
from understatapi import UnderstatClient

In [2]:
TEAM_NAME = "Tottenham"
SEASONS = ["2022", "2023", "2024", "2025"]

In [3]:
player_stats = []
with UnderstatClient() as understat:

    try:
        for season in SEASONS: 
            players = understat.team(team=TEAM_NAME).get_player_data(season=season)
            df_players = pd.DataFrame(players)
            df_players["season"] = season
            player_stats.append(df_players)
            
    except Exception as e:
        print(f"Error fetching data for season {season}: {e}")

df_raw = pd.concat(player_stats, ignore_index=True)

In [4]:
df = df_raw.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            120 non-null    str  
 1   player_name   120 non-null    str  
 2   games         120 non-null    str  
 3   time          120 non-null    str  
 4   goals         120 non-null    str  
 5   xG            120 non-null    str  
 6   assists       120 non-null    str  
 7   xA            120 non-null    str  
 8   shots         120 non-null    str  
 9   key_passes    120 non-null    str  
 10  yellow_cards  120 non-null    str  
 11  red_cards     120 non-null    str  
 12  position      120 non-null    str  
 13  team_title    120 non-null    str  
 14  npg           120 non-null    str  
 15  npxG          120 non-null    str  
 16  xGChain       120 non-null    str  
 17  xGBuildup     120 non-null    str  
 18  season        120 non-null    str  
dtypes: str(19)
memory usage: 17.9 KB


In [5]:
# エラーが出たらその列をそのまま返す関数
def safe_to_numeric(column):
    try:
        return pd.to_numeric(column, errors="raise")
    except ValueError:
        return column

In [6]:
df = df.apply(safe_to_numeric)

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            120 non-null    int64  
 1   player_name   120 non-null    str    
 2   games         120 non-null    int64  
 3   time          120 non-null    int64  
 4   goals         120 non-null    int64  
 5   xG            120 non-null    float64
 6   assists       120 non-null    int64  
 7   xA            120 non-null    float64
 8   shots         120 non-null    int64  
 9   key_passes    120 non-null    int64  
 10  yellow_cards  120 non-null    int64  
 11  red_cards     120 non-null    int64  
 12  position      120 non-null    str    
 13  team_title    120 non-null    str    
 14  npg           120 non-null    int64  
 15  npxG          120 non-null    float64
 16  xGChain       120 non-null    float64
 17  xGBuildup     120 non-null    float64
 18  season        120 non-null    int64  
dty

In [14]:
# 総得点とxGの差分
df["xG_diff"] = df["goals"] - df["xG"]
# ゴール・コンパージョン・レート
df["goal_conversion_rate"] = df["goals"] / df["shots"] * 100
# 1シュートあたりの平均期待値
df["xG_per_shot"] = df["xG"] / df["shots"]
# 1ゴールにかかる時間
df["minutes_per_goal"] = df["time"] / df["goals"]
# infと-infをNaNに置き換え
df["minutes_per_goal"] = df["minutes_per_goal"].replace([np.inf, -np.inf], np.nan)
# 90分あたりの得点数
df["goals_per_90"] = df["goals"] / df["time"] * 90

In [16]:
cols_to_keep = ["season", "id", "player_name", "games", "shots", "goals", "xG", "xG_diff", "xG_per_shot", "goal_conversion_rate", "goals_per_90", "minutes_per_goal"]

In [20]:
df[cols_to_keep].head()

,season,id,player_name,games,shots,goals,xG,xG_diff,xG_per_shot,goal_conversion_rate,goals_per_90,minutes_per_goal
0,2022,647,Harry Kane,38,133,30,23.064440,6.935560,0.173417,22.556391,0.790861,113.800000
1,2022,453,Son Heung-Min,36,81,10,9.597895,0.402105,0.118493,12.345679,0.308219,292.000000
2,2022,6108,Rodrigo Bentancur,18,15,5,2.436944,2.563056,0.162463,33.333333,0.296834,303.200000
3,2022,343,Pierre-Emile Højbjerg,35,33,4,1.529552,2.470448,0.046350,12.121212,0.114759,784.250000
4,2022,6912,Pedro Porro,15,25,3,1.397329,1.602671,0.055893,12.000000,0.233969,384.666667
